In [10]:
# IrisPathQ Route Optimizer v0.2

#Classical preprocessing for quantum route optimization

#- Fuel and Distance calculations
#- Cost matrix construction

import subprocess
import os

print("IrisPathQ Route Optimizer v0.2")
print("Setting up C compilation environment...\n")

# gcc verification
result = subprocess.run(["gcc", "--version"], capture_output=True, text=True)
print(result.stdout.split('\n')[0])
print("\nReady for next step")

IrisPathQ Route Optimizer v0.2
Setting up C compilation environment...

gcc (GCC) 15.1.0

Ready for next step


In [65]:
%%writefile data_structures.h
/**
 * IrisPathQ Route Optimizer
 * Data structures for flights, routes, and optimization
*/

#ifndef DATA_STRUCTURES_H
#define DATA_STRUCTURES_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

#define MAX_FLIGHTS 500
#define MAX_WAYPOINTS 1000
#define MAX_ROUTES_PER_FLIGHT 10
#define MAX_WAYPOINTS_PER_ROUTE 50
#define EARTH_RADIUS_KM 6371.0
#define M_PI 3.14159265358979323846

typedef struct {
    char id[10];
    char name[50];
    double latitude;
    double longitude;
} Waypoint;

typedef struct {
    double latitude;
    double longitude;
    double wind_speed;
    double wind_direction;
    char weather_type[20];
} Weather;

typedef struct {
    char type[10];
    double cruise_speed;
    double fuel_burn_rate;
    double max_range;
} AircraftType;

typedef struct {
    char flight_id[10];
    char origin[10];
    char destination[10];
    char departure_time[10];
    AircraftType aircraft;
} Flight;

typedef struct {
    int waypoint_indices[MAX_WAYPOINTS_PER_ROUTE];
    int num_waypoints;
    double total_distance;
    double fuel_cost;
    double time_cost;
    int conflict_count;
} Route;

typedef struct {
    Flight flights[MAX_FLIGHTS];
    int num_flights;
    
    Waypoint waypoints[MAX_WAYPOINTS];
    int num_waypoints;
    
    Weather weather_cells[100];
    int num_weather_cells;
    
    Route routes[MAX_FLIGHTS][MAX_ROUTES_PER_FLIGHT];
    int num_routes_per_flight[MAX_FLIGHTS];
} ProblemInstance;

double calculate_distance(double lat1, double lon1, double lat2, double lon2);
double calculate_fuel(Route *route, AircraftType *aircraft);
void print_route(Route *route, Waypoint *waypoints);
int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells);

int load_flights(const char *filename, ProblemInstance *problem);
int load_waypoints(const char *filename, ProblemInstance *problem);
int load_weather(const char *filename, ProblemInstance *problem);
int generate_alternative_routes(ProblemInstance *problem, int flight_idx, int num_routes);
void build_cost_matrix(ProblemInstance *problem, double **cost_matrix, int *matrix_size);
void export_cost_matrix(double *cost_matrix, int matrix_size, const char *filename);
void export_full_matrix(double *cost_matrix, int matrix_size, const char *filename);

#endif

Overwriting data_structures.h


In [55]:
%%writefile utils.c
/**
 * IrisPathQ Route Optimizer
 * Distance and fuel calculations
*/

#include "data_structures.h"

double calculate_distance(double lat1, double lon1, double lat2, double lon2) {
    //degree to radian conversion
    double lat1_rad = lat1 * M_PI / 180.0;
    double lat2_rad = lat2 * M_PI / 180.0;
    double delta_lat = (lat2 - lat1) * M_PI / 180.0;
    double delta_lon = (lon2 - lon1) * M_PI / 180.0;

    //haversineformula 
    double a = sin(delta_lat / 2.0) * sin(delta_lat / 2.0) +
               cos(lat1_rad) * cos(lat2_rad) *
               sin(delta_lon / 2.0) * sin(delta_lon / 2.0);
    
    double c = 2.0 * atan2(sqrt(a), sqrt(1.0 - a));
    double distance_km = EARTH_RADIUS_KM * c;
    
    return distance_km / 1.852;
}

double calculate_fuel(Route *route, AircraftType *aircraft) {
    double time_hours = route->total_distance / aircraft->cruise_speed;
    return time_hours * aircraft->fuel_burn_rate;
}

// For conflicts due to weather, but rigth now no conflicts-- weather.csv has no thunderstorm as input
int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells) {
    for (int i = 0; i < num_cells; i++) {
        if (strcmp(weather_cells[i].weather_type, "THUNDERSTORM") == 0) {
            double dist = calculate_distance(lat, lon, 
                                            weather_cells[i].latitude, 
                                            weather_cells[i].longitude);
            if (dist < 50.0) {
                return 1;
            }
        }
    }
    return 0;
}

void print_route(Route *route, Waypoint *waypoints) {
    printf("Route: ");
    for (int i = 0; i < route->num_waypoints; i++) {
        printf("%s ", waypoints[route->waypoint_indices[i]].id);
        if (i < route->num_waypoints - 1) printf("-> ");
    }
    printf("\n");
    printf("Distance: %.2f nm, Fuel: %.2f kg, Time: %.2f hrs\n",
           route->total_distance, route->fuel_cost, route->time_cost);
}

Writing utils.c


In [56]:
import subprocess

print("Testing C compilation...\n")

result = subprocess.run(["gcc","-c", "utils.c", "-o", "utils.o"], capture_output = True ,text = True)

if result.returncode == 0: 
 print("Compilation works\n")
else:
 print("compilation failed\n")
 print(result.stderr)



Testing C compilation...

Compilation works



In [57]:
%%writefile data_loader.c
/**
 * IrisPathQ Route Optimizer
 * Load flights, waypoints, and weather data from CSV files
*/

#include "data_structures.h"

int load_flights(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_flights = 0;
    while (fgets(line, sizeof(line), file) && problem->num_flights < MAX_FLIGHTS) {
        Flight *f = &problem->flights[problem->num_flights];
        
        sscanf(line, "%[^,],%[^,],%[^,],%[^,],%s",
               f->flight_id, f->origin, f->destination, 
               f->departure_time, f->aircraft.type);
        
        if (strcmp(f->aircraft.type, "B737") == 0) {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2500.0;
            f->aircraft.max_range = 3000.0;
        } else if (strcmp(f->aircraft.type, "A320") == 0) {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2400.0;
            f->aircraft.max_range = 3200.0;
        } else {
            f->aircraft.cruise_speed = 450.0;
            f->aircraft.fuel_burn_rate = 2500.0;
            f->aircraft.max_range = 3000.0;
        }
        
        problem->num_flights++;
    }
    
    fclose(file);
    printf("Loaded %d flights\n", problem->num_flights);
    return problem->num_flights;
}

int load_waypoints(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_waypoints = 0;
    while (fgets(line, sizeof(line), file) && problem->num_waypoints < MAX_WAYPOINTS) {
        Waypoint *w = &problem->waypoints[problem->num_waypoints];
        
        sscanf(line, "%[^,],%lf,%lf,%s",
               w->id, &w->latitude, &w->longitude, w->name);
        
        problem->num_waypoints++;
    }
    
    fclose(file);
    printf("Loaded %d waypoints\n", problem->num_waypoints);
    return problem->num_waypoints;
}

int load_weather(const char *filename, ProblemInstance *problem) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open %s\n", filename);
        return -1;
    }
    
    char line[256];
    fgets(line, sizeof(line), file);
    
    problem->num_weather_cells = 0;
    while (fgets(line, sizeof(line), file) && problem->num_weather_cells < 100) {
        Weather *w = &problem->weather_cells[problem->num_weather_cells];
        
        sscanf(line, "%lf,%lf,%lf,%lf,%s",
               &w->latitude, &w->longitude, &w->wind_speed, 
               &w->wind_direction, w->weather_type);
        
        problem->num_weather_cells++;
    }
    
    fclose(file);
    printf("Loaded %d weather cells\n", problem->num_weather_cells);
    return problem->num_weather_cells;
}

Overwriting data_loader.c


In [58]:
%%writefile route_optimizer.c
/**
 * IrisPathQ Route Optimizer
 * Generate alternative routes and build cost matrix both full and sparse for now
*/

#include "data_structures.h"
#include <float.h>

int generate_alternative_routes(ProblemInstance *problem, int flight_idx, int num_routes) {
    Flight *flight = &problem->flights[flight_idx];
    
    int origin_idx = -1, dest_idx = -1;
    for (int i = 0; i < problem->num_waypoints; i++) {
        if (strcmp(problem->waypoints[i].id, flight->origin) == 0) {
            origin_idx = i;
        }
        if (strcmp(problem->waypoints[i].id, flight->destination) == 0) {
            dest_idx = i;
        }
    }
    
    if (origin_idx == -1 || dest_idx == -1) {
        printf("  Error: Waypoints not found for %s\n", flight->flight_id);
        return -1;
    }
    
    printf("  %s (%s -> %s): ", flight->flight_id, flight->origin, flight->destination);
    
    double distance = calculate_distance(
        problem->waypoints[origin_idx].latitude,
        problem->waypoints[origin_idx].longitude,
        problem->waypoints[dest_idx].latitude,
        problem->waypoints[dest_idx].longitude
    );
    
    for (int r = 0; r < num_routes && r < MAX_ROUTES_PER_FLIGHT; r++) {
        Route *route = &problem->routes[flight_idx][r];
        
        route->num_waypoints = 2;
        route->waypoint_indices[0] = origin_idx;
        route->waypoint_indices[1] = dest_idx;
        
        route->total_distance = distance;
        
        if (r > 0) {
            route->total_distance *= (1.0 + (r * 0.02));
        }
        
        route->fuel_cost = calculate_fuel(route, &flight->aircraft);
        route->time_cost = route->total_distance / flight->aircraft.cruise_speed;
        route->conflict_count = 0;
        
        problem->num_routes_per_flight[flight_idx]++;
    }
    
    printf("%d routes\n", num_routes);
    return problem->num_routes_per_flight[flight_idx];
}

void build_cost_matrix(ProblemInstance *problem, double **cost_matrix, int *matrix_size) {
    int total_vars = 0;
    for (int i = 0; i < problem->num_flights; i++) {
        total_vars += problem->num_routes_per_flight[i];
    }
    
    *matrix_size = total_vars;
    *cost_matrix = (double*)calloc(total_vars * total_vars, sizeof(double));
    
    printf("\n  Building cost matrix (%d x %d)...\n", total_vars, total_vars);
    
    int var_index = 0;
    for (int flight = 0; flight < problem->num_flights; flight++) {
        for (int route = 0; route < problem->num_routes_per_flight[flight]; route++) {
            double fuel_cost = problem->routes[flight][route].fuel_cost;
            (*cost_matrix)[var_index * total_vars + var_index] = fuel_cost;
            var_index++;
        }
    }
    
    printf("  Cost matrix built\n");
}

void export_cost_matrix(double *cost_matrix, int matrix_size, const char *filename) {
    FILE *file = fopen(filename, "w");
    if (!file) {
        printf("  Cannot open %s\n", filename);
        return;
    }
    
    fprintf(file, "%d\n", matrix_size);
    
    for (int i = 0; i < matrix_size; i++) {
        for (int j = 0; j < matrix_size; j++) {
            double value = cost_matrix[i * matrix_size + j];
            if (value != 0.0) {
                fprintf(file, "%d,%d,%.2f\n", i, j, value);
            }
        }
    }
    
    fclose(file);
    printf("  Exported to %s\n", filename);
}

void export_full_matrix(double *cost_matrix, int matrix_size, const char *filename) {
    FILE *file = fopen(filename, "w");
    if (!file) {
        printf("  Cannot open %s\n", filename);
        return;
    }
    
    fprintf(file, "Full Cost Matrix (%d x %d)\n", matrix_size, matrix_size);
    fprintf(file, "================================\n\n");
    fprintf(file, "  Diagonal values = Fuel cost (kg)\n");
    fprintf(file, "  Off-diagonal = 0 (no conflicts)\n\n");
    fprintf(file, " (0,0) is flight 0 taking route 0 its fuel cost, where rows 0 to 4 is flight routes fro the zeroth flight(UA123), similary col 0 to 4 is allroutes available for Flight 0  \n");
    fprintf(file, "  CONFLICT = Route conflict penalty; No conflicts for now (10000+ kg)\n");
    fprintf(file, "  0 = No interaction\n\n");
    fprintf(file, "\n");
    fprintf(file, "      ");
    for (int j = 0; j < matrix_size; j++) {
        fprintf(file, "Col%-5d ", j);
    }
    fprintf(file, "\n");
    
    for (int i = 0; i < matrix_size; i++) {
        fprintf(file, "Row%-2d | ", i);
        for (int j = 0; j < matrix_size; j++) {
            double value = cost_matrix[i * matrix_size + j];
            if (value == 0.0) {
                fprintf(file, "    0    ");
            } else {
                fprintf(file, "%8.0f ", value);
            }
        }
        fprintf(file, "\n");
    }
    
    fclose(file);
    printf("  Full matrix exported to %s\n", filename);
}

Overwriting route_optimizer.c


In [59]:
%%writefile main.c
/**
 * IrisPathQ Route Optimizer
 * Main program - data loading, route generation, and cost matrix creation
*/

#include "data_structures.h"

int main() {
    printf("====================================\n");
    printf("IrisPathQ Route Optimization\n");
    printf("====================================\n\n");
    
    ProblemInstance problem;
    memset(&problem, 0, sizeof(ProblemInstance));
    
    printf("Loading data...\n");
    load_flights("data/flights.csv", &problem);
    load_waypoints("data/waypoints.csv", &problem);
    load_weather("data/weather.csv", &problem);
    
    printf("\n====================================\n");
    printf("Generating Routes\n");
    printf("====================================\n\n");
    
    int num_alternative_routes = 5;
    for (int i = 0; i < problem.num_flights; i++) {
        generate_alternative_routes(&problem, i, num_alternative_routes);
    }
    
    printf("\n====================================\n");
    printf("Building Cost Matrix\n");
    printf("====================================\n\n");
    
    double *cost_matrix = NULL;
    int matrix_size = 0;
    build_cost_matrix(&problem, &cost_matrix, &matrix_size);
    
    export_cost_matrix(cost_matrix, matrix_size, "output/cost_matrix.txt");
    export_full_matrix(cost_matrix, matrix_size, "output/full_matrix.txt");
    
    printf("\n====================================\n");
    printf("Summary\n");
    printf("====================================\n");
    printf("Total flights: %d\n", problem.num_flights);
    printf("Total routes generated: %d\n", matrix_size);
    printf("Cost matrix size: %d x %d\n", matrix_size, matrix_size);
    printf("\nReady for quantum optimization!\n");
    
    free(cost_matrix);
    return 0;
}

Overwriting main.c


In [60]:
import subprocess

print("Compiling IrisPathQ Route Optimizer...\n")

result = subprocess.run([
    "gcc", "-o", "route_optimizer","main.c", "data_loader.c", "utils.c", "route_optimizer.c","-lm"], capture_output=True, text=True)

if result.returncode == 0:
    print("Compilation working\nExecutable: route_optimizer")

else:
    print("Compilation failed:")
    print(result.stderr)

Compiling IrisPathQ Route Optimizer...

Compilation working
Executable: route_optimizer


In [61]:
import subprocess

print("Running IrisPathQ Route Optimizer...\n")
print("=" * 50) #interesting

result = subprocess.run(["./route_optimizer"], capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print("Error:", result.stderr)

Running IrisPathQ Route Optimizer...

IrisPathQ Route Optimization

Loading data...
Loaded 5 flights
Loaded 10 waypoints
Loaded 3 weather cells

Generating Routes

  AC101 (YYZ -> YVR): 5 routes
  WS202 (YUL -> YYC): 5 routes
  AC303 (YEG -> YOW): 5 routes
  WS404 (YWG -> YHZ): 5 routes
  AC505 (YYJ -> YYT): 5 routes

Building Cost Matrix


  Building cost matrix (25 x 25)...
  Cost matrix built
  Exported to output/cost_matrix.txt
  Full matrix exported to output/full_matrix.txt

Summary
Total flights: 5
Total routes generated: 25
Cost matrix size: 25 x 25

Ready for quantum optimization!



In [46]:
print("Cost Matrix:\n")
with open("output/cost_matrix.txt", "r") as f:
    lines = f.readlines()
    for line in lines:
        print(line.strip())

Cost Matrix:

25
0,0,10035.91
1,1,10236.63
2,2,10437.35
3,3,10638.07
4,4,10838.79
5,5,8655.23
6,6,8828.33
7,7,9001.44
8,8,9174.54
9,9,9347.65
10,10,8547.33
11,11,8718.28
12,12,8889.22
13,13,9060.17
14,14,9231.12
15,15,7729.46
16,16,7884.04
17,17,8038.63
18,18,8193.22
19,19,8347.81
20,20,15151.09
21,21,15454.11
22,22,15757.13
23,23,16060.15
24,24,16363.17


In [62]:
print("Full Matrix Visualization (first 30 lines only ):\n")
with open("output/full_matrix.txt", "r") as f:
    lines = f.readlines()[:30]
    for line in lines:
        print(line.rstrip())

Full Matrix Visualization (first 30 lines only ):

Full Cost Matrix (25 x 25)

  Diagonal values = Fuel cost (kg)
  Off-diagonal = 0 (no conflicts)

 (0,0) is flight 0 taking route 0 its fuel cost, where rows 0 to 4 is flight routes fro the zeroth flight(UA123), similary col 0 to 4 is allroutes available for Flight 0
  CONFLICT = Route conflict penalty; No conflicts for now (10000+ kg)
  0 = No interaction


      Col0     Col1     Col2     Col3     Col4     Col5     Col6     Col7     Col8     Col9     Col10    Col11    Col12    Col13    Col14    Col15    Col16    Col17    Col18    Col19    Col20    Col21    Col22    Col23    Col24
Row0  |    10036     0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0        0
Row1  |     0       10237     0        0        0        0        0        0        0        0        0        0        0        0